# 🔭 Lab 2.3 — Build an Evaluation Dataset from Production Traces

## Learning Objectives
In this notebook, you will learn:
1. **Traced Agent** — Wrap an LLM call with `@mlflow.trace` so every invocation lands in your experiment
2. **Trace Search** — Use `mlflow.search_traces(filter_string=...)` to retrieve recent successful runs
3. **Trace → Dataset** — Add traces to a UC dataset via `merge_records(traces=...)` — inputs/outputs auto-extract
4. **Production Filtering** — Apply `attributes.status = 'OK'` and `attributes.name = ...` to scope the eval set cleanly
5. **Annotation Pattern** — Add `expected_facts` after the fact so the Correctness judge can grade trace-derived rows

## Prerequisites
- Lab 1.3 completed — namespace and experiment exist
- Foundation Model API endpoint reachable from the cluster


In [ ]:
# ============================================================================
# 📦 INSTALL PACKAGES
# ============================================================================

%pip install --quiet "mlflow[databricks]>=3.1" databricks-openai

dbutils.library.restartPython()


---
## Step 1 — Configure Namespace and MLflow Experiment


In [ ]:
# ============================================================================
# 🗂️ STEP 1 - CONFIGURE NAMESPACE AND MLFLOW EXPERIMENT
# ============================================================================

import mlflow

CATALOG       = "genai_eval_tutorial"
SCHEMA        = "module_01"
DATASET_TABLE = "tutorial_eval_from_traces_v1"

UC_SCHEMA   = f"{CATALOG}.{SCHEMA}"
DATASET_FQN = f"{UC_SCHEMA}.{DATASET_TABLE}"

USER_EMAIL = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)
EXPERIMENT_PATH = f"/Users/{USER_EMAIL}/genai-eval-tutorial"
mlflow.set_experiment(EXPERIMENT_PATH)

print(f"Experiment: {EXPERIMENT_PATH}")
print(f"Target dataset: {DATASET_FQN}")


---
## Step 2 — Build a Tiny Traced Agent

We need something to *produce* traces. We wrap a Foundation Model API call with `@mlflow.trace` so each invocation lands in the experiment with `inputs` and `outputs` automatically extracted.

The `name` argument is important — we'll filter on it later when searching traces, so production noise doesn't pollute our eval dataset.


In [ ]:
# ============================================================================
# 🔭 STEP 2 - BUILD A TINY TRACED AGENT
# ============================================================================

from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
mlflow.openai.autolog()

@mlflow.trace(name="my_agent")
def my_agent(question: str) -> str:
    resp = client.chat.completions.create(
        model="databricks-claude-sonnet-4",
        messages=[
            {"role": "system", "content": "You are a concise Databricks expert. Answer in 2 sentences."},
            {"role": "user",   "content": question},
        ],
    )
    return resp.choices[0].message.content


---
## Step 3 — Simulate Production Traffic (Generate 10 Traces)

Each call below produces one trace. We use a fixed list of representative questions so the lab is reproducible.


In [ ]:
# ============================================================================
# 🤖 STEP 3 - SIMULATE PRODUCTION TRAFFIC (GENERATE 10 TRACES)
# ============================================================================

import time

questions = [
    "What is Z-ordering in Delta Lake?",
    "Explain the VACUUM command.",
    "What does Auto Loader do?",
    "How does Unity Catalog secure tables?",
    "What is Photon?",
    "When should I use Delta Live Tables vs plain Delta?",
    "What is the medallion architecture?",
    "How does time travel work in Delta?",
    "What is MLflow Tracing for?",
    "Explain Databricks SQL Serverless.",
]

# Capture the start timestamp BEFORE the agent runs so we don't pull in older traces
run_start_ms = int(time.time() * 1000)

for q in questions:
    print(my_agent(q)[:80] + "...")

print(f"\nGenerated {len(questions)} traces.")


---
## Step 4 — Retrieve the Recent, Successful Traces

The recommended filter for building eval datasets:
- `attributes.status = 'OK'` — drop failed runs
- `attributes.timestamp_ms > <recent>` — keep the window tight
- filter by trace name to scope to *one* agent (critical in shared experiments)


In [ ]:
# ============================================================================
# 🔭 STEP 4 - RETRIEVE THE RECENT, SUCCESSFUL TRACES
# ============================================================================

# Window: from just before this lab started, to now
window_start_ms = run_start_ms - 60_000  # 1-minute safety buffer

traces = mlflow.search_traces(
    experiment_names=[EXPERIMENT_PATH],
    filter_string=(
        f"attributes.timestamp_ms > {window_start_ms}"
        f" AND attributes.status = 'OK'"
        f" AND attributes.name = 'my_agent'"
    ),
    order_by=["attributes.timestamp_ms DESC"],
)

print(f"Retrieved {len(traces)} traces.")
display(traces)


---
## Step 5 — Create the UC Dataset and Merge the Traces

`merge_records(traces=...)` auto-extracts the agent's input arguments and return value into the `inputs` and `outputs` columns. No hand-mapping required.


In [ ]:
# ============================================================================
# 🔭 STEP 5 - CREATE THE UC DATASET AND MERGE THE TRACES
# ============================================================================

import mlflow.genai.datasets

eval_dataset = mlflow.genai.datasets.create_dataset(name=DATASET_FQN)
eval_dataset = eval_dataset.merge_records(traces=traces)

print(f"Dataset {eval_dataset.name} now has {eval_dataset.to_df().count()} rows.")


---
## Step 6 — Inspect the Auto-Extracted Columns

Notice how `inputs` carries the `question` arg from the function signature, and `outputs` carries the model's reply. This is exactly the schema `mlflow.genai.evaluate()` expects.


In [ ]:
# ============================================================================
# 🔍 STEP 6 - INSPECT THE AUTO-EXTRACTED COLUMNS
# ============================================================================

display(eval_dataset.to_df())


In [ ]:
# ============================================================================
# ▶️ VIEW AS A PLAIN DELTA TABLE TOO
# ============================================================================

display(spark.table(DATASET_FQN))


---
## Step 7 — (Optional) Add `expected_facts` Manually for the Correctness Judge

Datasets built from traces start without ground truth. To enable the **Correctness** judge, a domain expert (or another labelling step) needs to add `expectations`. Below we annotate the first row inline as a demonstration.


In [ ]:
# ============================================================================
# 🧩 STEP 7 - (OPTIONAL) ADD `EXPECTED_FACTS` MANUALLY FOR THE CORRECTNESS JUDGE
# ============================================================================

# Pull the first row's request_id so we can re-merge with expectations
first_row = eval_dataset.to_df().limit(1).collect()[0]
request_id = first_row["request_id"] if "request_id" in first_row.asDict() else None

if request_id:
    eval_dataset = eval_dataset.merge_records(records=[{
        "inputs": first_row["inputs"],
        "expectations": {
            "expected_facts": ["data skipping", "co-locates", "query performance"]
        },
    }])
    print("Annotated row 1 with expected_facts.")
else:
    print("Skipping annotation — schema differs from this MLflow version.")


---
## Lab Complete

| Check | Status |
| --- | --- |
| 10 traces generated under the `my_agent` name | ✅ |
| `search_traces` returned only recent + OK + `my_agent` traces | ✅ |
| UC dataset `tutorial_eval_from_traces_v1` populated | ✅ |
| `inputs` and `outputs` auto-extracted from traces | ✅ |

Next: **Lab 2.4** — let Databricks AI synthesise a diverse eval set straight from your documents.


---
## 📝 Summary

In this notebook, we covered:

### 1. Why Build From Traces
- **Real distribution.** Synthetic Q&A rarely matches the messy queries you actually serve.
- **Regression coverage.** Each trace becomes a regression test for behavior you've already shipped.

### 2. Filter Hygiene
- **`status = 'OK'`** drops failed runs that would poison the eval set.
- **`name = 'my_agent'`** scopes to one agent in shared experiments.
- **`timestamp_ms > <recent>`** keeps the window tight — capture the cut-off *before* the agent runs.

### 3. Auto-Extraction
- **`merge_records(traces=...)`** maps function arguments → `inputs` and return value → `outputs` with no hand-mapping.
- Trace-derived rows arrive without `expectations` — you must add ground truth before the Correctness judge will work.

### Next Steps
- **Lab 2.4** — let Databricks AI synthesize a diverse eval set straight from your documents.
